In [9]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_squared_error
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')


In [10]:
print("Loading datasets...")
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
submission_df = pd.read_csv('sample_submission.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

Loading datasets...
Train shape: (77299, 11)
Test shape: (41778, 10)


In [11]:
def preprocess_data(df, is_train=True):
    df_copy = df.copy()
    
    df_copy['Hour'] = df_copy['timestamp'].apply(lambda x: int(str(x).split(':')[0]))
    df_copy['Minute'] = df_copy['timestamp'].apply(lambda x: int(str(x).split(':')[1]))
    df_copy['Time_Fraction'] = df_copy['Hour'] + df_copy['Minute'] / 60.0
    
    df_copy['Hour_sin'] = np.sin(2 * np.pi * df_copy['Hour'] / 24.0)
    df_copy['Hour_cos'] = np.cos(2 * np.pi * df_copy['Hour'] / 24.0)
    
    df_copy['RoadType'] = df_copy['RoadType'].fillna('Unknown')
    df_copy['Temperature'] = df_copy['Temperature'].fillna(df_copy['Temperature'].median() if is_train else 20.0)
    df_copy['Weather'] = df_copy['Weather'].fillna('Unknown')
    
    categorical_cols = ['RoadType', 'LargeVehicles', 'Landmarks', 'Weather']
    for col in categorical_cols:
        le = LabelEncoder()
        df_copy[col] = le.fit_transform(df_copy[col].astype(str))
        
    geohash_le = LabelEncoder()
    df_copy['geohash_encoded'] = geohash_le.fit_transform(df_copy['geohash'].astype(str))
    
    features_to_drop = ['Index', 'geohash', 'timestamp']
    df_copy = df_copy.drop(columns=features_to_drop)
    
    return df_copy


In [12]:
print("Preprocessing and engineering features...")
processed_train = preprocess_data(train_df, is_train=True)
processed_test = preprocess_data(test_df, is_train=False)

# Separate features and target variable
X = processed_train.drop(columns=['demand'])
y = processed_train['demand']
X_test = processed_test.copy()

print("Features selected for training:", list(X.columns))

Preprocessing and engineering features...
Features selected for training: ['day', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather', 'Hour', 'Minute', 'Time_Fraction', 'Hour_sin', 'Hour_cos', 'geohash_encoded']


In [13]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training the XGBoost Regressor...")
model = xgb.XGBRegressor(
    n_estimators=1000,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    tree_method='hist',
    early_stopping_rounds=50
)

Training the XGBoost Regressor...


In [14]:
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100
)

[0]	validation_0-rmse:0.13773
[100]	validation_0-rmse:0.05533
[200]	validation_0-rmse:0.05402
[300]	validation_0-rmse:0.05368
[395]	validation_0-rmse:0.05365


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=50,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=1000,
             n_jobs=None, num_parallel_tree=None, ...)

In [15]:
val_preds = model.predict(X_val)

r2 = r2_score(y_val, y_val_preds := np.clip(val_preds, 0, None)) # Demand cannot be negative
rmse = np.sqrt(mean_squared_error(y_val, y_val_preds))

print("\n=== Validation Performance ===")
print(f"R2 Score (Variance Explained): {r2 * 100:.2f}%")
print(f"Root Mean Squared Error (RMSE): {rmse:.5f}")


=== Validation Performance ===
R2 Score (Variance Explained): 85.80%
Root Mean Squared Error (RMSE): 0.05361


In [16]:
print("Generating final test predictions...")
test_preds = model.predict(X_test)
test_preds = np.clip(test_preds, 0, None)  # Post-processing safeguard

# Create submission payload
submission = pd.DataFrame({
    'Index': test_df['Index'],
    'demand': test_preds
})

submission.to_csv('final_submission.csv', index=False)
print("Submission file 'final_submission.csv' successfully saved!")

Generating final test predictions...


Submission file 'final_submission.csv' successfully saved!
